### ACU

In [6]:
from datetime import datetime
import logging
import os
import sys
from typing import Any, Optional
from dotenv import find_dotenv, load_dotenv
from pathlib import Path

# Add the parent directory to the Python path to import the sample_helper module
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'python'))
sys.path.insert(0, str(Path().resolve()))
sys.path.insert(0, str(Path("notebooks/02-azure-content-understanding").resolve()))

from dotenv import load_dotenv
load_dotenv() 

from content_understanding_client import AzureContentUnderstandingClient
from sample_helper import save_json_to_file 
from azure.identity import DefaultAzureCredential

load_dotenv(find_dotenv())
logging.basicConfig(level=logging.INFO)

# For authentication, you can use either token-based auth or subscription key; only one is required
AZURE_AI_ENDPOINT = os.getenv("AZURE_AI_ENDPOINT")
# IMPORTANT: Replace with your actual subscription key or set it in your ".env" file if not using token authentication
AZURE_AI_API_KEY = os.getenv("AZURE_AI_API_KEY")
API_VERSION = "2025-11-01"

# Create token provider for Azure AD authentication
def token_provider():
    credential = DefaultAzureCredential()
    token = credential.get_token("https://cognitiveservices.azure.com/.default")
    return token.token

# Create the Content Understanding client
try:
    client = AzureContentUnderstandingClient(
        endpoint=AZURE_AI_ENDPOINT,
        api_version=API_VERSION,
        subscription_key=AZURE_AI_API_KEY,
        token_provider=token_provider if not AZURE_AI_API_KEY else None,
        x_ms_useragent="azure-ai-content-understanding-python-sample-ga"    # The user agent is used for tracking sample usage and does not provide identity information. You can change this if you want to opt out of tracking.
    )
    credential_type = "Subscription Key" if AZURE_AI_API_KEY else "Azure AD Token"
    print(f"✅ Client created successfully")
    print(f"   Endpoint: {AZURE_AI_ENDPOINT}")
    print(f"   Credential: {credential_type}")
    print(f"   API Version: {API_VERSION}")
except Exception as e:
    credential_type = "Subscription Key" if AZURE_AI_API_KEY else "Azure AD Token"
    print(f"❌ Failed to create client")
    print(f"   Endpoint: {AZURE_AI_ENDPOINT}")
    print(f"   Credential: {credential_type}")
    print(f"   Error: {e}")
    raise

✅ Client created successfully
   Endpoint: https://azure-foundry-westus-resource.services.ai.azure.com/
   Credential: Subscription Key
   API Version: 2025-11-01


### Blob Storage

In [1]:
import os
import json
import threading
from enum import Enum
from pathlib import PurePosixPath
from typing import Optional, Dict, Any
from datetime import datetime

from azure.storage.blob import BlobServiceClient, BlobClient
from azure.core.exceptions import ResourceExistsError


class Stage(Enum):
    RAW = "raw"               # original pdf
    ACU = "acu"               # full ACU response json
    FIELDS = "fields"         # slim fields-only json for DB/eval/bbox
    ANNOTATED = "annotated"   # annotated artifacts (optional)


class BlobStorage:
    _instance: Optional["BlobStorage"] = None
    _lock = threading.Lock()

    def __new__(cls):
        with cls._lock:
            if cls._instance is None:
                cls._instance = super().__new__(cls)
        return cls._instance

    def __init__(self):
        if getattr(self, "_initialized", False):
            return

        self._connection_string = None
        self._blob_service_client = None

        self._initialized_containers = set()
        self._container_lock = threading.Lock()
        self._initialized = True

    @property
    def connection_string(self) -> str:
        # Prefer env var
        cs = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
        if not cs:
            raise ValueError("AZURE_STORAGE_CONNECTION_STRING not set.")
        return cs

    @property
    def blob_service_client(self) -> BlobServiceClient:
        if self._blob_service_client is None:
            self._blob_service_client = BlobServiceClient.from_connection_string(self.connection_string)
        return self._blob_service_client

    def _ensure_container_exists(self, container_name: str) -> None:
        if container_name in self._initialized_containers:
            return

        with self._container_lock:
            if container_name in self._initialized_containers:
                return

            try:
                cc = self.blob_service_client.get_container_client(container_name)
                cc.create_container()
            except ResourceExistsError:
                pass

            self._initialized_containers.add(container_name)

    def ensure_all_containers_ready(self) -> None:
        for stage in Stage:
            self._ensure_container_exists(stage.value)

    def blob_path(self, doc_id: str, stage: Stage, ext: str) -> PurePosixPath:
        if not ext.startswith("."):
            ext = "." + ext
        return PurePosixPath(f"{doc_id}{ext}")

    def blob_client(self, doc_id: str, stage: Stage, ext: str) -> BlobClient:
        container_name = stage.value
        self._ensure_container_exists(container_name)
        path = self.blob_path(doc_id, stage, ext)
        container_client = self.blob_service_client.get_container_client(container_name)
        return container_client.get_blob_client(str(path))

    def upload_bytes(self, doc_id: str, stage: Stage, ext: str, data: bytes, overwrite: bool = True) -> None:
        bc = self.blob_client(doc_id, stage, ext)
        bc.upload_blob(data, overwrite=overwrite)

    def download_bytes(self, doc_id: str, stage: Stage, ext: str) -> Optional[bytes]:
        try:
            bc = self.blob_client(doc_id, stage, ext)
            return bc.download_blob().readall()
        except Exception:
            return None

    def upload_json(self, doc_id: str, stage: Stage, ext: str, payload: Dict[str, Any], overwrite: bool = True) -> None:
        wrapper = {
            "document_id": doc_id,
            "stage": stage.value,
            "timestamp": datetime.utcnow().isoformat() + "Z",
            "payload": payload,
        }
        data = json.dumps(wrapper, indent=2, ensure_ascii=False).encode("utf-8")
        self.upload_bytes(doc_id, stage, ext, data, overwrite=overwrite)

    def download_json(self, doc_id: str, stage: Stage, ext: str) -> Optional[Dict[str, Any]]:
        b = self.download_bytes(doc_id, stage, ext)
        if not b:
            return None
        return json.loads(b.decode("utf-8"))

    def list_blobs_in_stage(self, stage: Stage) -> list[str]:
        self._ensure_container_exists(stage.value)
        container_client = self.blob_service_client.get_container_client(stage.value)
        return [b.name for b in container_client.list_blobs()]


def get_storage() -> BlobStorage:
    return BlobStorage()


In [2]:
from typing import Tuple

def split_fields(raw_acu_result: dict) -> Tuple[dict, dict, dict]:
    """
    Returns: (raw_fields, normalized_fields, usage_summary)
    """
    content = raw_acu_result["result"]["contents"][0]
    fields = content.get("fields", {})

    raw_fields = {k: v for k, v in fields.items() if k.endswith("_raw")}
    normalized_fields = {k: v for k, v in fields.items() if k.endswith("_normalized")}

    usage = raw_acu_result.get("result", {}).get("usage") or raw_acu_result.get("usage") or {}
    usage_summary = {
        "documentPagesStandard": usage.get("documentPagesStandard"),
        "contextualizationTokens": usage.get("contextualizationTokens"),
        "tokens": usage.get("tokens", {})
    }

    return raw_fields, normalized_fields, usage_summary


In [3]:
import uuid

storage = get_storage()
storage.ensure_all_containers_ready()

In [4]:
doc_id = str(uuid.uuid4())
pdf_path = "data/AlliedEsportsEntertainmentInc_20190815_8-K_EX-10.19_11788293_EX-10.19_Content License Agreement.pdf"

In [5]:
with open(pdf_path, "rb") as f:
    storage.upload_bytes(doc_id, Stage.RAW, ".pdf", f.read())

In [7]:
license_analyzer_id = "license_agreement_extraction_wrt_CUAD_v4_raw_normalized_singlepass"

analysis_response = client.begin_analyze_binary(
    analyzer_id=license_analyzer_id,
    file_location=pdf_path,
)

INFO:content_understanding_client:Analyzing binary file data/AlliedEsportsEntertainmentInc_20190815_8-K_EX-10.19_11788293_EX-10.19_Content License Agreement.pdf with analyzer: license_agreement_extraction_wrt_CUAD_v4_raw_normalized_singlepass


In [8]:
acu_result = client.poll_result(analysis_response)

INFO:content_understanding_client:Request 847837c6-42a7-4260-817a-1141ce57960f in progress ...
INFO:content_understanding_client:Request 847837c6-42a7-4260-817a-1141ce57960f in progress ...
INFO:content_understanding_client:Request 847837c6-42a7-4260-817a-1141ce57960f in progress ...
INFO:content_understanding_client:Request 847837c6-42a7-4260-817a-1141ce57960f in progress ...
INFO:content_understanding_client:Request 847837c6-42a7-4260-817a-1141ce57960f in progress ...
INFO:content_understanding_client:Request 847837c6-42a7-4260-817a-1141ce57960f in progress ...
INFO:content_understanding_client:Request 847837c6-42a7-4260-817a-1141ce57960f in progress ...
INFO:content_understanding_client:Request 847837c6-42a7-4260-817a-1141ce57960f in progress ...
INFO:content_understanding_client:Request 847837c6-42a7-4260-817a-1141ce57960f in progress ...
INFO:content_understanding_client:Request 847837c6-42a7-4260-817a-1141ce57960f in progress ...
INFO:content_understanding_client:Request 847837c6

In [9]:
storage.upload_json(doc_id, Stage.ACU, ".json", acu_result)

C:\Users\deril\AppData\Local\Temp\ipykernel_4928\3637314652.py:102: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat() + "Z",
INFO:azure.core.pipeline.policies.http_logging_policy:Request URL: 'https://rg13286088654.blob.core.windows.net/acu/9ddb4884-9915-470d-a813-7d747b3c6a1e.json'
Request method: 'PUT'
Request headers:
    'Content-Length': '55827'
    'x-ms-blob-type': 'REDACTED'
    'x-ms-version': 'REDACTED'
    'Content-Type': 'application/octet-stream'
    'Accept': 'application/xml'
    'User-Agent': 'azsdk-python-storage-blob/12.21.0 Python/3.13.4 (Windows-11-10.0.26200-SP0)'
    'x-ms-date': 'REDACTED'
    'x-ms-client-request-id': '56a95994-073f-11f1-b959-e08f4ce88e78'
    'Authorization': 'REDACTED'
A body is sent with the request
INFO:azure.core.pipeline.policies.http_logging_po

In [10]:
raw_fields, normalized_fields, usage_summary = split_fields(acu_result)

In [11]:
fields_payload = {
    "analyzerId": acu_result["result"].get("analyzerId"),
    "apiVersion": acu_result["result"].get("apiVersion"),
    "raw_fields": raw_fields,
    "normalized_fields": normalized_fields,
    "usage": usage_summary,
}

In [12]:
storage.upload_json(doc_id, Stage.FIELDS, ".json", fields_payload)

C:\Users\deril\AppData\Local\Temp\ipykernel_4928\3637314652.py:102: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat() + "Z",
INFO:azure.core.pipeline.policies.http_logging_policy:Request URL: 'https://rg13286088654.blob.core.windows.net/fields/9ddb4884-9915-470d-a813-7d747b3c6a1e.json'
Request method: 'PUT'
Request headers:
    'Content-Length': '14930'
    'x-ms-blob-type': 'REDACTED'
    'x-ms-version': 'REDACTED'
    'Content-Type': 'application/octet-stream'
    'Accept': 'application/xml'
    'User-Agent': 'azsdk-python-storage-blob/12.21.0 Python/3.13.4 (Windows-11-10.0.26200-SP0)'
    'x-ms-date': 'REDACTED'
    'x-ms-client-request-id': '85e51e82-073f-11f1-927d-e08f4ce88e78'
    'Authorization': 'REDACTED'
A body is sent with the request
INFO:azure.core.pipeline.policies.http_logging

#### Retrieval example

In [13]:
storage = get_storage()

fields_doc = storage.download_json(doc_id, Stage.FIELDS, ".json")
payload = fields_doc["payload"]

print("Normalized keys:", list(payload["normalized_fields"].keys())[:5])
print("Raw keys:", list(payload["raw_fields"].keys())[:5])
print("Usage:", payload["usage"])

INFO:azure.core.pipeline.policies.http_logging_policy:Request URL: 'https://rg13286088654.blob.core.windows.net/fields/9ddb4884-9915-470d-a813-7d747b3c6a1e.json'
Request method: 'GET'
Request headers:
    'x-ms-range': 'REDACTED'
    'x-ms-version': 'REDACTED'
    'Accept': 'application/xml'
    'User-Agent': 'azsdk-python-storage-blob/12.21.0 Python/3.13.4 (Windows-11-10.0.26200-SP0)'
    'x-ms-date': 'REDACTED'
    'x-ms-client-request-id': '8d7100f1-073f-11f1-ac67-e08f4ce88e78'
    'Authorization': 'REDACTED'
No body was attached to the request
INFO:azure.core.pipeline.policies.http_logging_policy:Response status: 206
Response headers:
    'Content-Length': '14930'
    'Content-Type': 'application/octet-stream'
    'Content-Range': 'REDACTED'
    'Last-Modified': 'Wed, 11 Feb 2026 11:48:03 GMT'
    'Accept-Ranges': 'REDACTED'
    'ETag': '"0x8DE69636A0474F1"'
    'Server': 'Windows-Azure-Blob/1.0 Microsoft-HTTPAPI/2.0'
    'x-ms-request-id': 'ad5e87b3-001e-0043-394c-9b14bd000000'
  

Normalized keys: ['DocumentName_normalized', 'Parties_normalized', 'AgreementDate_normalized', 'EffectiveDate_normalized', 'ExpirationDate_normalized']
Raw keys: ['DocumentName_raw', 'Parties_raw', 'AgreementDate_raw', 'EffectiveDate_raw', 'ExpirationDate_raw']
Usage: {'documentPagesStandard': 10, 'contextualizationTokens': 10000, 'tokens': {'gpt-4.1-mini-input': 16331, 'gpt-4.1-mini-output': 1868}}
